# Train Classifier + Extract Pairs

Self-contained workflow for training the resolved-thread classifier from `threads.jsonl`, then extracting `confirmed_pairs.jsonl`.


## Bootstrap Repository


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, urllib.request

REPO_URL = 'https://github.com/eftpmc/reddibase.git'
REPO_DIR = Path.home() / 'reddibase'

def in_repo(path):
    return (path / 'requirements.txt').exists() and (path / 'framework').exists()

cwd = Path.cwd()
if in_repo(cwd):
    ROOT = cwd
else:
    if not REPO_DIR.exists():
        subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_DIR)])
    ROOT = REPO_DIR

os.chdir(ROOT)
print(f'Repo root: {Path.cwd()}')
print(f'Python: {sys.executable}')


## Install Dependencies


In [ ]:
# Datacenter CUDA setup: install a PyTorch build compatible with CUDA 12.4,
# then install the rest of the project deps without letting pip upgrade torch.
%pip uninstall -y torch torchvision torchaudio
%pip install torch==2.5.1 --index-url https://download.pytorch.org/whl/cu124
%pip install -r requirements.txt


## GPU Check


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA version:', torch.version.cuda)
else:
    print('No CUDA GPU found. This notebook will work, but full training will be slow.')


## Configuration


In [ ]:
MODEL = 'tipofmyjoystick'
THREADS = Path(f'data/converted/{MODEL}/threads.jsonl')
AUDIT = Path(f'data/converted/{MODEL}/thread_audit.json')
CLASSIFIER_DIR = Path(f'models/{MODEL}/resolved_thread_classifier')
SMOKE_CLASSIFIER_DIR = Path(f'models/{MODEL}/resolved_thread_classifier_smoke')
PAIRS = Path(f'models/{MODEL}/confirmed_pairs.jsonl')
SMOKE_PAIRS = Path(f'models/{MODEL}/confirmed_pairs_smoke.jsonl')
EPOCHS = 3
BATCH_SIZE = 32
GRAD_ACCUM = 1
print('Threads:', THREADS)
print('Classifier:', CLASSIFIER_DIR)
print('Pairs:', PAIRS)


## Acquire Threads Artifact

This notebook requires `threads.jsonl`. If it is not already in the repo, set `THREADS_SOURCE` to a local path or URL and run this cell.


In [ ]:
# If threads.jsonl is not already present, set THREADS_SOURCE to a local path or URL.
# Examples:
# THREADS_SOURCE = '/scratch/yourname/threads.jsonl'
# THREADS_SOURCE = 'https://your-storage/threads.jsonl'
THREADS_SOURCE = ''

THREADS.parent.mkdir(parents=True, exist_ok=True)
if not THREADS.exists() and THREADS_SOURCE:
    if THREADS_SOURCE.startswith(('http://', 'https://')):
        print(f'Downloading {THREADS_SOURCE} -> {THREADS}')
        urllib.request.urlretrieve(THREADS_SOURCE, THREADS)
    else:
        src = Path(THREADS_SOURCE).expanduser()
        if not src.exists():
            raise FileNotFoundError(f'THREADS_SOURCE does not exist: {src}')
        print(f'Copying {src} -> {THREADS}')
        shutil.copyfile(src, THREADS)

if not THREADS.exists():
    raise FileNotFoundError(
        f'Missing {THREADS}. Upload/copy threads.jsonl to this path, '
        'or set THREADS_SOURCE above and rerun this cell.'
    )

print(f'Threads artifact: {THREADS.resolve()}')
print(f'Size: {THREADS.stat().st_size / (1024**3):.2f} GB')


## Audit Training Signals


In [ ]:
!python -m scripts.audit_threads {MODEL} --threads {THREADS} --output {AUDIT}


## View Audit Summary


In [ ]:
import json
report = json.loads(AUDIT.read_text(encoding='utf-8'))
{
    'threads': report['threads'],
    'messages': report['messages'],
    'usable_weak_answer_threads': report['usable_weak_answer_threads'],
    'excluded_weak_answer_threads': report['excluded_weak_answer_threads'],
    'top_usable_weak_answers': report['top_usable_weak_answers'][:10],
}


## Optional Smoke Training

Use this on a new machine if you want a quick dependency/CUDA check before the full run.


In [ ]:
# !python -m scripts.train_classifier {MODEL} --threads {THREADS} --limit 1000 --epochs 1 --batch-size 4 --grad-accum 2 --output {SMOKE_CLASSIFIER_DIR}


## Full Classifier Training


In [ ]:
!python -m scripts.train_classifier {MODEL} --threads {THREADS} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --grad-accum {GRAD_ACCUM} --output {CLASSIFIER_DIR}


## Optional Pair Extraction Smoke Test


In [ ]:
# !python -m scripts.build_dataset {MODEL} --threads {THREADS} --classifier {CLASSIFIER_DIR} --limit 5000 --output {SMOKE_PAIRS}


## Full Pair Extraction


In [ ]:
!python -m scripts.build_dataset {MODEL} --threads {THREADS} --classifier {CLASSIFIER_DIR} --output {PAIRS}


## Pair Count


In [ ]:
if not PAIRS.exists():
    raise FileNotFoundError(PAIRS)
pair_count = sum(1 for _ in PAIRS.open(encoding='utf-8'))
print(f'Confirmed pairs: {pair_count:,}')
